In [ ]:
import pandas as pd
import geopandas as gpd
import seaborn as sns
from sqlalchemy import create_engine
from dotenv import load_dotenv
import matplotlib.pyplot as plt
import numpy as np
import os
import matplotlib.pyplot as plt
import contextily as ctx

load_dotenv()

DB_NAME=os.getenv('DB_NAME')
DB_USER=os.getenv('DB_USER')
DB_PW=os.getenv('DB_PW')
DB_HOST=os.getenv('DB_HOST')
DB_PORT=os.getenv('DB_PORT')

ENGINE = create_engine(f"postgresql://{DB_USER}:{DB_PW}@{DB_HOST}:{DB_PORT}/{DB_NAME}")
ENGINE_NOMINATIM = create_engine(f"postgresql://nominatim:qaIACxO6wMR3@{DB_HOST}:5555/nominatim")

## IDPC

In [ ]:
def get_df_from_query(query):
    return pd.read_sql(sql=query, con=ENGINE)

def get_gdf_from_query(query, geom_col = 'geometry'):
    return gpd.read_postgis(sql=query, con=ENGINE, geom_col=geom_col)

In [ ]:
CRIMES_DICT = {
    "homicidio": "Homicídio Doloso",
    "latrocinio": "Latrocínio",
    "hom_por_intervencao_policial": "Morte por Intervenção Policial",
    "tentat_hom": "Tentativa de Homicídio",
    "lesao_corp_dolosa": "Lesão Corporal Dolosa",
    "estupro": "Estupro",
    "sequestro": "Sequestro",
    "extorsao": "Extorsão",
    "estelionato": "Estelionato",
    "trafico_drogas": "Tráfico Drogas",
    "policiais_mortos": "Policiais Mortos em Serviço",
    "lesao_corp_morte": "Lesão Corporal Seguida de Morte",
    "hom_culposo": "Homicídio Culposo",
    "lesao_corp_culposa": "Lesão Corporal Culposa",
    "ameaca": "Ameaça",
    "roubo_transeunte": "Roubo a Transeunte",
    "roubo_celular": "Roubo de Celular",
    "roubo_em_coletivo": "Roubo em Coletivo",
    "roubo_veiculo": "Roubo de Veículo",
    "roubo_carga": "Roubo de Carga",
    "roubo_residencia": "Roubo a Residência",
    "roubo_banco": "Roubo a Banco",
    "roubo_comercio": "Roubo de Comércio",
    "roubo_cx_eletronico": "Roubo a Caixa Eletrônico",
    "roubo_conducao_saque": "Roubo com Condução a Saque",
    "roubo_apos_saque": "Roubo após Saque",
    "roubo_bicicleta": "Roubo de Bicicleta",
    "outros_roubos": "Outros Roubos",
    "furto_veiculos": "Furto de Veículos",
    "furto_celular": "Furto de Celular",
    "furto_transeunte": "Furto a Transeunte",
    "furto_coletivo": "Furto em Coletivo",
    "furto_bicicleta": "Furto de Bicicleta",
    "outros_furtos": "Outros Furtos",
    "sequestro_relampago": "Sequestro Relâmpago",
    "posse_drogas": "Posse Drogas",
    "pessoas_desaparecidas": "Pessoas Desaparecidas",
    "encontro_cadaver": "Encontro Cadáver",
    'tiros': 'Tiros'
}


CRIMES_DICT_INV = {v: k for k, v in CRIMES_DICT.items()}

In [ ]:
len(CRIMES_DICT)

In [ ]:
query_crimes_percebidos = """
select c.nome_dp, c2.nome_crime
from tcc.cisp c
left join
	tcc.bairro b on c.id = b.cisp_id
left join
	tcc.fato_comentario_percepcao fcp on fcp.bairro_id_bairro = b.id_bairro
left join
	tcc.crime c2 on fcp.crime_id_crime = c2.id_crime 
where c2.id_crime <= 40
union all
select c.nome_dp, c2.nome_crime
from tcc.cisp c
left join
	tcc.bairro b on c.id = b.cisp_id
left join
	tcc.fato_publicacao_percepcao fcp on fcp.bairro_id_bairro = b.id_bairro
left join
	tcc.crime c2 on fcp.crime_id_crime = c2.id_crime 
where c2.id_crime <= 40
"""

df_crimes_percebidos_cisp = pd.read_sql(query_crimes_percebidos, con=ENGINE)
df_crimes_percebidos_cisp

In [ ]:
df_crimes_percebidos_cisp['nome_crime'] = df_crimes_percebidos_cisp['nome_crime'].map(CRIMES_DICT_INV)
df_crimes_percebidos_cisp

In [ ]:
df_contagem = df_crimes_percebidos_cisp.groupby(['nome_dp', 'nome_crime']).size().reset_index(name='total_mencoes')
df_contagem

In [ ]:
df_vetor_percep = df_contagem.pivot(
    index='nome_dp', 
    columns='nome_crime', 
    values='total_mencoes'
).fillna(0)
df_vetor_percep

In [ ]:
for crime in CRIMES_DICT.keys():
    if crime not in df_vetor_percep.columns:
        df_vetor_percep[crime] = 0.0
df_vetor_percep = df_vetor_percep[CRIMES_DICT.keys()]
df_vetor_percep

In [ ]:
df_vetor_percep['vetor_perc'] = df_vetor_percep.apply(
    lambda row: np.array(row.values, dtype=float), 
    axis=1
)
df_vetor_percep

In [ ]:
df_vetor_percep.loc[:, 'vetor_perc_normalizado'] = df_vetor_percep['vetor_perc'].apply(lambda x: x / x.sum()) # normalizacao
df_vetor_percep

In [ ]:
df_isp = pd.read_sql("""
WITH cte_isp AS (
    SELECT 
        c.nome_dp,
        COALESCE(SUM(homicidio), 0) AS homicidio,
        COALESCE(SUM(latrocinio), 0) AS latrocinio,
        COALESCE(SUM(hom_por_intervencao_policial), 0) AS hom_por_intervencao_policial,
        COALESCE(SUM(tentat_hom), 0) AS tentat_hom,
        COALESCE(SUM(lesao_corp_dolosa), 0) AS lesao_corp_dolosa,
        COALESCE(SUM(estupro), 0) AS estupro,
        COALESCE(SUM(sequestro), 0) AS sequestro,
        COALESCE(SUM(extorsao), 0) AS extorsao,
        COALESCE(SUM(estelionato), 0) AS estelionato,
        COALESCE(SUM(trafico_drogas), 0) AS trafico_drogas,
        COALESCE(SUM(policiais_mortos), 0) AS policiais_mortos,
        COALESCE(SUM(lesao_corp_morte), 0) AS lesao_corp_morte,
        COALESCE(SUM(hom_culposo), 0) AS hom_culposo,
        COALESCE(SUM(lesao_corp_culposa), 0) AS lesao_corp_culposa,
        COALESCE(SUM(ameaca), 0) AS ameaca,
        COALESCE(SUM(roubo_transeunte), 0) AS roubo_transeunte,
        COALESCE(SUM(roubo_celular), 0) AS roubo_celular,
        COALESCE(SUM(roubo_em_coletivo), 0) AS roubo_em_coletivo,
        COALESCE(SUM(roubo_veiculo), 0) AS roubo_veiculo,
        COALESCE(SUM(roubo_carga), 0) AS roubo_carga,
        COALESCE(SUM(roubo_residencia), 0) AS roubo_residencia,
        COALESCE(SUM(roubo_banco), 0) AS roubo_banco,
        COALESCE(SUM(roubo_comercio), 0) AS roubo_comercio,
        COALESCE(SUM(roubo_cx_eletronico), 0) AS roubo_cx_eletronico,
        COALESCE(SUM(roubo_conducao_saque), 0) AS roubo_conducao_saque,
        COALESCE(SUM(roubo_apos_saque), 0) AS roubo_apos_saque,
        COALESCE(SUM(roubo_bicicleta), 0) AS roubo_bicicleta,
        COALESCE(SUM(outros_roubos), 0) AS outros_roubos,
        COALESCE(SUM(furto_veiculos), 0) AS furto_veiculos,
        COALESCE(SUM(furto_celular), 0) AS furto_celular,
        COALESCE(SUM(furto_transeunte), 0) AS furto_transeunte,
        COALESCE(SUM(furto_coletivo), 0) AS furto_coletivo,
        COALESCE(SUM(furto_bicicleta), 0) AS furto_bicicleta,
        COALESCE(SUM(outros_furtos), 0) AS outros_furtos,
        COALESCE(SUM(sequestro_relampago), 0) AS sequestro_relampago,
        COALESCE(SUM(posse_drogas), 0) AS posse_drogas,
        COALESCE(SUM(pessoas_desaparecidas), 0) AS pessoas_desaparecidas,
        COALESCE(SUM(encontro_cadaver), 0) AS encontro_cadaver
    FROM tcc.cisp c
    LEFT JOIN tcc.registro_cisp rc ON c.id = rc.cisp_id
    WHERE c.nome_dp != 'Não identificado'
    GROUP BY c.nome_dp
),
cte_fc AS (
    SELECT 
        c.nome_dp, 
        COUNT(ot.id) AS tiros
    FROM ocorrencia_tiroteio ot
    LEFT JOIN tcc.bairro b ON ot.bairro_id_bairro = b.id_bairro
    LEFT JOIN tcc.cisp c ON c.id = b.cisp_id
    GROUP BY c.nome_dp
)
SELECT 
    base.nome_dp,
    ci.homicidio,
    ci.latrocinio,
    ci.hom_por_intervencao_policial,
    ci.tentat_hom,
    ci.lesao_corp_dolosa,
    ci.estupro,
    ci.sequestro,
    ci.extorsao,
    ci.estelionato,
    ci.trafico_drogas,
    ci.policiais_mortos,
    ci.lesao_corp_morte,
    ci.hom_culposo,
    ci.lesao_corp_culposa,
    ci.ameaca,
    ci.roubo_transeunte,
    ci.roubo_celular,
    ci.roubo_em_coletivo,
    ci.roubo_veiculo,
    ci.roubo_carga,
    ci.roubo_residencia,
    ci.roubo_banco,
    ci.roubo_comercio,
    ci.roubo_cx_eletronico,
    ci.roubo_conducao_saque,
    ci.roubo_apos_saque,
    ci.roubo_bicicleta,
    ci.outros_roubos,
    ci.furto_veiculos,
    ci.furto_celular,
    ci.furto_transeunte,
    ci.furto_coletivo,
    ci.furto_bicicleta,
    ci.outros_furtos,
    ci.sequestro_relampago,
    ci.posse_drogas,
    ci.pessoas_desaparecidas,
    ci.encontro_cadaver,
    COALESCE(cf.tiros, 0) AS tiros
FROM tcc.cisp base
LEFT JOIN cte_isp ci ON base.nome_dp = ci.nome_dp
LEFT JOIN cte_fc cf ON base.nome_dp = cf.nome_dp
WHERE base.nome_dp != 'Não identificado';
""", con=ENGINE, index_col='nome_dp')

df_isp

In [ ]:
df_isp['vetor_real'] = df_isp.apply(
    lambda row: np.array(row.values, dtype=float), 
    axis=1
)
df_isp['vetor_real_normalizado'] = df_isp['vetor_real'].apply(lambda x: x/x.sum())
df_isp

In [ ]:
df_final = df_vetor_percep[['vetor_perc_normalizado']].merge(df_isp[['vetor_real_normalizado']], left_index=True, right_index=True)
df_final

In [ ]:
import numpy as np
import pandas as pd
from scipy.spatial.distance import cosine

# Lista de crimes na ordem correta
CRIMES_LIST = list(CRIMES_DICT.values())  # valores legíveis dos crimes

def analisar_desalinhamento(row):
    # 1. Extrair Vetores
    v_perc = row['vetor_perc_normalizado']
    v_real = row['vetor_real_normalizado']

    # Evitar vetores nulos
    if np.sum(v_perc) == 0 or np.sum(v_real) == 0:
        distancia = 0.0 
    else:
        distancia = cosine(v_real, v_perc)
    
    # 3. Calcular Deltas (Diferença pontual por crime)
    deltas = v_perc - v_real
    
    # Crime Mais Superestimado (Maior diferença positiva)
    idx_super = np.argmax(deltas)
    val_super = deltas[idx_super]
    
    # Crime Mais Subestimado (Maior diferença negativa)
    idx_sub = np.argmin(deltas)
    val_sub = deltas[idx_sub]
    
    # Crime Mais Alinhado (Menor distância absoluta)
    idx_fiel = np.argmin(np.abs(deltas))
    val_fiel = deltas[idx_fiel]
    
    return pd.Series({
        'indice_desalinhamento': distancia,
        'crime_superestimado': CRIMES_LIST[idx_super],
        'delta_super_pp': val_super,
        'crime_subestimado': CRIMES_LIST[idx_sub],
        'delta_sub_pp': val_sub,
        'crime_mais_fiel': CRIMES_LIST[idx_fiel],
        'delta_fiel_pp': val_fiel
    })

# Aplicando a função
df_resultados = df_final.join(
    df_final.apply(analisar_desalinhamento, axis=1)
)

df_resultados

In [ ]:
from sqlalchemy.dialects.postgresql import ARRAY
from sqlalchemy import Float

df_resultados['vetor_perc_normalizado'] = df_resultados['vetor_perc_normalizado'].apply(lambda x: x.tolist())
df_resultados['vetor_real_normalizado'] = df_resultados['vetor_real_normalizado'].apply(lambda x: x.tolist())

In [ ]:
df_resultados

In [ ]:
df_resultados.to_sql(
    "resultados_vetores",
    con=ENGINE,
    if_exists="replace",
    schema='tcc',
    dtype={"vetor_perc_normalizado": ARRAY(Float), "vetor_real_normalizado": ARRAY(Float)}
)

## Logradouros

In [ ]:
df_logradouros = pd.read_sql("""select nome_bairro, logradouro, nome_autor, nome_crime, count(*) as qtd_mencoes
                             from tcc.crimes_nome_autor
                             where (logradouro is not null) or (logradouro != 'None')
                             group by 1, 2, 3, 4
                             order by 5 desc""", con=ENGINE)
gdf_logradouros = gpd.GeoDataFrame(df_logradouros)
gdf_logradouros

In [ ]:
def busca_geometria_logradouro(bairro: str, logradouro: str):
    logradouro = logradouro.replace("'", "''")
    sql = f"""
        with cte as (SELECT l.name -> 'name' AS nome, ST_Union(l.geometry) AS geometry
FROM placex l
JOIN placex p
  ON ST_Within(l.geometry, p.geometry)
join placex p2
	on p.parent_place_id = p2.place_id
WHERE p.name -> 'name' = '{bairro}'
  AND p.class = 'boundary'
  AND l.class in ('highway', 'leisure')
  AND l.name IS NOT null
  and p2.name -> 'name' = 'Rio de Janeiro'
  and l.name -> 'name' = '{logradouro}'
GROUP BY 1
UNION
SELECT p.name -> 'name' AS nome, p.geometry
FROM placex p
JOIN placex p2
  ON p.parent_place_id = p2.place_id
join placex p3
	on p2.parent_place_id = p3.place_id
WHERE p2.name -> 'name' = '{bairro}'
  AND p2.class = 'boundary'
  AND p.class in ('highway', 'leisure')
  AND p.name IS NOT null
  and p3.name -> 'name' = 'Rio de Janeiro'
  and p.name -> 'name' = '{logradouro}')
select nome, st_union(geometry) as geom
from cte
group by 1;
    """

    gdf = gpd.read_postgis(sql, ENGINE_NOMINATIM, geom_col='geom')
    if gdf.empty or gdf.geometry.isnull().all():
        return None

    return gdf.geometry.iloc[0]  # retorna Shapely MultiLineString ou LineString

busca_geometria_logradouro(bairro='Curicica', logradouro='Rua João Bruno Lobo')

In [ ]:
gdf_logradouros['geom'] = gdf_logradouros.apply(
    lambda row: busca_geometria_logradouro(bairro=row['nome_bairro'], logradouro=row['logradouro']),
    axis=1
)
gdf_logradouros = gdf_logradouros.set_geometry('geom')
gdf_logradouros

In [ ]:
if gdf_logradouros.crs is None:
    gdf_logradouros = gdf_logradouros.set_crs(epsg=4326)

gdf_logradouros = gdf_logradouros.to_crs(epsg=3857)

ax = gdf_logradouros.plot(figsize=(12,12), color='blue', linewidth=2, alpha=0.7)

ctx.add_basemap(ax, source=ctx.providers.OpenStreetMap.Mapnik)

plt.axis('off')
plt.show()

In [ ]:
gdf_final = gdf_logradouros[~gdf_logradouros.geometry.isna()]
gdf_final

In [ ]:
gdf_final.to_postgis("linhas_logradouros", con=ENGINE, schema='tcc', if_exists='replace')